The point of this notebook is to:
- determine the length of time it will take to use libraries to process text detection.
- test optimisation strategies.

Aims/requirements:
- All profanity should be filtered out, particularly those in subtitles. Starred words should also be removed.
- Profanity filtering should only take ~1 second per second of footage. This ensures an hour video can be filtered in ~1 hour
- Ideally, this process should take < 0.5 seconds per second of footage.

In [1]:
## Imports and global constants
import cv2
import easyocr, pytesseract, paddleocr
import time

video_path = "../data/raw/example1.mp4"

/home/ananth/repos/video-content-filter/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/ananth/repos/video-content-filter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Load the profanity file

In [2]:
# Load profanity
profanity_set = set()
profanity_file_path = "../src/content_filter/config/profanity_words.txt"
with open(profanity_file_path, "r", encoding="utf-8") as f:
    for line in f:
        word = line.strip().lower()
        profanity_set.add(word)

def contains_profanity(text):
    words = text.lower().split()
    return any(word.strip(".,!?") in profanity_set for word in words)

Define a function for benchmarking EasyOCR

In [ ]:
def run_easy_ocr_test(duration):
    reader = easyocr.Reader(['en'], gpu=False, quantize=True)
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"FPS: {fps}")

    frame_count = 0

    profanity_boxes = []

    max_frames = int(fps * duration)  # first 3 seconds
    t0 = time.time()
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # Run OCR on frame
        results = reader.readtext(frame)

        for (bbox, text, confidence) in results:
            if contains_profanity(text):
                # Convert bbox points to int
                bbox = [(int(x), int(y)) for (x, y) in bbox]
                profanity_boxes.append((frame_count, bbox))

                # Draw rectangle on frame
                top_left = bbox[0]
                bottom_right = bbox[2]
                cv2.rectangle(frame, top_left, bottom_right, (0, 0, 255), 2)

        frame_count += 1

    t1 = time.time()
    cap.release()
    cv2.destroyAllWindows()
    return t1 - t0


In [4]:
# duration = 3
# dt = run_easy_ocr_test(duration)
# print(f"It took: {dt} seconds to process all of the footage")
# print(f"Time taken per second of footage: {(dt)/duration :.2f}s")

#### Findings for naive method using EasyOCR on CPU:
- Time taken per second of footage: 181.64s
- 3 minutes per second of footage is WAY too much time.
- Cannot use GPU as the project should run locally on a machine.

In [5]:
### Trying Tesseract ###
def run_tesseract_ocr(seconds, output_path):
    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    profanity_boxes = []

    fps = cap.get(cv2.CAP_PROP_FPS)
    max_frames = int(fps * seconds)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    t0 = time.time()
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # Get detailed OCR data (word-level boxes)
        data = pytesseract.image_to_data(frame, output_type=pytesseract.Output.DICT)

        n_boxes = len(data["text"])

        fourcc = cv2.VideoWriter.fourcc('m', 'p', '4', 'v')  # use 'XVID' if mp4v fails
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        for i in range(n_boxes):
            word = data["text"][i]
            conf = int(data["conf"][i])

            if conf > 60 and contains_profanity(word):
                x = data["left"][i]
                y = data["top"][i]
                w = data["width"][i]
                h = data["height"][i]

                profanity_boxes.append((frame_count, (x, y, w, h)))

                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

        frame_count += 1
        out.write(frame)

    t1 = time.time()
    cap.release()
    cv2.destroyAllWindows()

    return (t1-t0), n_boxes


In [6]:
# dt, _ = run_tesseract_ocr(3, "../data/processed/censored_text.mp4")

In [7]:
# print(f"It took: {dt}")
# print(f"Time taken per second of footage: {(dt)/3 :.2f}s")

#### Tesseract naive findings
- Time taken per second of footage: 8.85s on average
- Still not fast enough: a 1 hour video will take 9 hours to filter, which is still too long.
- Need to judge accuracy

##### Testing quality of approaches: naive
In theory, the naive approach should be the most accurate as we are running the OCR every frame.

In [8]:
def get_text_on_image(image_data):
    """
    Use Tesseract OCR to extract all text and bounding boxes from an image.
    """
    # Convert to grayscale for better OCR accuracy
    if len(image_data.shape) == 3:
        gray = cv2.cvtColor(image_data, cv2.COLOR_BGR2GRAY)
    else:
        gray = image_data

    data = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)

    results = []
    num_boxes = len(data["text"])
    for i in range(num_boxes):
        text = data["text"][i].strip()

        # Skip empty text
        if not text:
            continue

        results.append({
            "text": text,
            "bbox": (data["left"][i], data["top"][i], data["width"][i], data["height"][i]),
        })

    return results

# image_path = "../data/samples/Image1.png" 

# image = cv2.imread(image_path)
# assert image is not None, f"Could not load image from: {image_path}"

# results = get_text_on_image(image)

# print(f"Found {len(results)} text region(s):\n")
# for r in results:
#     x, y, w, h = r["bbox"]
#     print(f"  Text: {r['text']!r:30s} | BBox(x={x}, y={y}, w={w}, h={h})")

There were many more text regions that were missed, including a subtitle region, which contained large text. Tesseract OCR on its own is not accurate enough for the project. TesseractOCR was built for text recognition in documents, not for text with messy backgrounds. The same text was detected when detecting by character instead, so no performance improvement there.<br>
We need to use a more powerful model but on a smaller area of the image. 

The main models for scene text detection are PaddleOCR (optimised for CPU) and EasyOCR (optimised for GPU).<br>
PaddleOCR is estimated to give a ~3-4x speed up on CPU, so I will test its speed and performance. Also worth testing EasyOCR performance.

In [ ]:
def run_paddle_ocr_test(image_path):
    """
    Run PaddleOCR on the given image data and return detected text and time taken.
    """
    paddle_ocr = paddleocr.PaddleOCR(lang='en')
    
    t0 = time.time()
    results = paddle_ocr.predict(image_path)
    dt = time.time() - t0

    texts = []
    if results and results[0]:
        for line in results[0]:
            text, confidence = line[1]
            texts.append(text)

    return texts, dt

# image_path = "../data/samples/Image1.png"
# image = cv2.imread(image_path)
# assert image is not None, f"Could not load image from: {image_path}"

# texts, dt = run_paddle_ocr_test(image)
# print(f"Time taken: {dt:.4f}s")
# print(f"Detected {len(texts)} text region(s):")
# for t in texts:
#     print(f"  {t!r}")


/home/ananth/repos/video-content-filter/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ananth/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ananth/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ananth/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_serv

NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)


Unfortunately, PaddleOCR is having many issues despite for re-installing it. I will use EasyOCR for now with heavy optimisations due to time I have to spend on this.

In [ ]:
image_path = "../data/samples/Image1.png"
image = cv2.imread(image_path)
assert image is not None, f"Could not load image from: {image_path}"

texts, dt = run_paddle_ocr_test(image)
print(f"Time taken: {dt:.4f}s")
print(f"Detected {len(texts)} text region(s):")
for t in texts:
    print(f"  {t!r}")
